In [12]:
import os
import json
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load GPT-2 model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Add a padding token
tokenizer.pad_token = tokenizer.eos_token

def analyze_bias(text, original_text):
    try:
        prompt = (
            f"Analyze the following text for different types of biases and compare it to the original text:\n\n"
            f"Original Text: {original_text}\n\n"
            f"Text: {text}\n\n"
            f"1. Sentiment: Provide a summary of the sentiment, notable examples of biased language, and rate the bias from 0 (no bias) to 1 (extreme bias). Describe the sentiment bias in three words.\n"
            f"2. Framing: Describe how the text frames the topic, identify any bias in the framing, provide notable examples, and rate the bias from 0 (no bias) to 1 (extreme bias). Describe the framing bias in three words.\n"
            f"3. Representation: Examine the representation of different groups in the text, identify any biases in how these groups are portrayed, provide notable examples, and rate the representation bias from 0 (no bias) to 1 (extreme bias). Describe the representation bias in three words.\n"
            f"4. Omission: Identify any significant omissions or differences in information that might indicate bias in the text, provide notable examples, and rate the omission bias from 0 (no bias) to 1 (extreme bias). Describe the omission bias in three words.\n"
            f"Additionally, extract any dates and times mentioned in the text along with a brief description of the associated events."
        )

        inputs = tokenizer.encode(prompt, return_tensors='pt', truncation=True, max_length=512, padding='max_length')
        attention_mask = inputs != tokenizer.pad_token_id
        outputs = model.generate(inputs, attention_mask=attention_mask, max_length=1024, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"Error during GPT-2 analysis: {e}")
        return None

def extract_bias_scores_and_examples(analysis_text):
    try:
        lines = analysis_text.split('\n')
        scores = {
            "sentiment": {"score": 0.0, "examples": [], "description": ""},
            "framing": {"score": 0.0, "examples": [], "description": ""},
            "representation": {"score": 0.0, "examples": [], "description": ""},
            "omission": {"score": 0.0, "examples": [], "description": ""},
            "events": []
        }
        current_section = None

        for line in lines:
            line = line.strip()
            if line.startswith("1. Sentiment:"):
                current_section = "sentiment"
            elif line.startswith("2. Framing:"):
                current_section = "framing"
            elif line.startswith("3. Representation:"):
                current_section = "representation"
            elif line.startswith("4. Omission:"):
                current_section = "omission"
            elif "extract any dates and times" in line:
                current_section = "events"
            elif current_section and current_section in scores:
                if 'rate the bias from 0 (no bias) to 1 (extreme bias)' in line:
                    try:
                        score = float(line.split(':')[-1].strip())
                        score = max(0.0, min(1.0, score))  # Ensure the score is between 0 and 1
                        scores[current_section]["score"] = score
                    except ValueError:
                        continue
                elif 'Describe the' in line:
                    scores[current_section]["description"] = line.split(':')[-1].strip()
                elif 'example' in line.lower() or 'notable' in line.lower():
                    scores[current_section]["examples"].append(line)
            elif current_section == "events":
                if line:
                    scores["events"].append(line)
        
        return scores
    except Exception as e:
        print(f"Error during extraction of bias scores and examples: {e}")
        return {
            "sentiment": {"score": "N/A", "examples": [], "description": "N/A"},
            "framing": {"score": "N/A", "examples": [], "description": "N/A"},
            "representation": {"score": "N/A", "examples": [], "description": "N/A"},
            "omission": {"score": "N/A", "examples": [], "description": "N/A"},
            "events": []
        }

def process_documents(folder_path):
    results = {}
    files = [f for f in os.listdir(folder_path) if f.endswith('.txt') and '_1_' in f]
    total_files = len(files)

    for idx, filename in enumerate(files):
        file_path = os.path.join(folder_path, filename)
        original_file_path = file_path.replace('_1_', '_0_')
        
        if os.path.exists(original_file_path):
            with open(file_path, 'r', encoding='utf-8') as file:
                text_variant = file.read()
            with open(original_file_path, 'r', encoding='utf-8') as original_file:
                original_text = original_file.read()

            try:
                analysis = analyze_bias(text_variant, original_text)
                if analysis:
                    scores_and_examples = extract_bias_scores_and_examples(analysis)
                else:
                    scores_and_examples = {
                        "sentiment": {"score": "N/A", "examples": [], "description": "N/A"},
                        "framing": {"score": "N/A", "examples": [], "description": "N/A"},
                        "representation": {"score": "N/A", "examples": [], "description": "N/A"},
                        "omission": {"score": "N/A", "examples": [], "description": "N/A"},
                        "events": []
                    }
            except Exception as e:
                print(f"Error processing file {filename}: {e}")
                scores_and_examples = {
                    "sentiment": {"score": "N/A", "examples": [], "description": "N/A"},
                    "framing": {"score": "N/A", "examples": [], "description": "N/A"},
                    "representation": {"score": "N/A", "examples": [], "description": "N/A"},
                    "omission": {"score": "N/A", "examples": [], "description": "N/A"},
                    "events": []
                }

            results[filename] = scores_and_examples
        
        # Print progress
        print(f"Processed {idx + 1}/{total_files} files ({(idx + 1) / total_files * 100:.2f}%)")

    return results

def main():
    folder_path = r"C:\Users\Raphael\Projects\rbuchmueller.github.io\VAST2024\MC1\mc1_data\articles"
    
    results = process_documents(folder_path)
    
    output_file = r"C:\Users\Raphael\Projects\rbuchmueller.github.io\VAST2024\MC1\mc1_data\analysis_results.json"
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(results, file, ensure_ascii=False, indent=4)

if __name__ == "__main__":
    main()


Processed 1/84 files (1.19%)
Processed 2/84 files (2.38%)
Processed 3/84 files (3.57%)
Processed 4/84 files (4.76%)
Processed 5/84 files (5.95%)
Processed 6/84 files (7.14%)
Processed 7/84 files (8.33%)
Processed 8/84 files (9.52%)
Processed 9/84 files (10.71%)
Processed 10/84 files (11.90%)
Processed 11/84 files (13.10%)
Processed 12/84 files (14.29%)
Processed 13/84 files (15.48%)
Processed 14/84 files (16.67%)
Processed 15/84 files (17.86%)
Processed 16/84 files (19.05%)
Processed 17/84 files (20.24%)
Processed 18/84 files (21.43%)
Processed 19/84 files (22.62%)
Processed 20/84 files (23.81%)
Processed 21/84 files (25.00%)
Processed 22/84 files (26.19%)
Processed 23/84 files (27.38%)
Processed 24/84 files (28.57%)
Processed 25/84 files (29.76%)
Processed 26/84 files (30.95%)
Processed 27/84 files (32.14%)
Processed 28/84 files (33.33%)
Processed 29/84 files (34.52%)
Processed 30/84 files (35.71%)
Processed 31/84 files (36.90%)
Processed 32/84 files (38.10%)
Processed 33/84 files (39